# Validación de métodos espectrales para reconstrucción DSA

Este cuaderno compara varios registros BIS unilaterales y bilaterales cambiando únicamente el método de obtención del mapa tiempo-frecuencia:

- Welch
- `scipy.signal.spectrogram`
- Wavelet continua

El resto del flujo se mantiene igual que en la metodología actual de la aplicación:

- canal 1 en unilateral;
- canal 1 izquierdo y canal 3 derecho en bilateral;
- filtro pasa-altos causal según `LoFilter`;
- ventana de 2 s y avance de 1 s;
- `detrend="constant"`;
- rango 0.5-30 Hz;
- bins de 0.5 Hz;
- suavizado causal tipo rolling con `SpSmooth`;
- shift fijo del flujo actual: 10 s unilateral y 6 s bilateral;
- máscaras comunes de calidad;
- comparación contra `.f_a` en escala dB, sin z-score.

Para que el cuaderno sea manejable, se evalúa solo el tramo inicial de cada registro (`MAX_SECONDS`).

In [1]:

# Imports y configuración del entorno.
# Añadimos la carpeta de la aplicación (APP_ROOT) al sys.path para poder
# reutilizar directamente sus funciones de lectura y reconstrucción, en
# lugar de reescribirlas o duplicarlas en el cuaderno.

from pathlib import Path
import sys

import numpy as np
import pandas as pd

from scipy.signal import spectrogram
from scipy.stats import pearsonr, spearmanr
import pywt

from IPython.display import display, Markdown

APP_ROOT = Path(r"C:\Users\usuario\OneDrive - Universidad de Burgos\Documentos\tfg\aplicaciones\visualizador_bis_icca")
if str(APP_ROOT) not in sys.path:
    sys.path.insert(0, str(APP_ROOT))

# Funciones y parámetros reutilizados de la aplicación (mismo flujo que en producción)
from src.lectura_spa import cargar_spa_unilateral_desde_ruta, cargar_spa_bilateral_desde_ruta
from src.lectura_fa import (
    cargar_fa_unilateral_desde_ruta,
    cargar_fa_bilateral_completo_desde_ruta,
    cargar_tiempos_fa_desde_ruta,
)
from src.alineacion_temporal import calcular_timeline_comun
from src.reconstruccion import (
    PARAMETROS_RECONSTRUCCION,
    extraer_parametros_eeg_desde_ruta,
    leer_inicio_ta_desde_ruta,
    _leer_raw_intercalado_desde_ruta,
    _extraer_suavizado_spsmooth,
    _extraer_filtro_lofilter,
    _calcular_indices_alineacion,
    _calcular_tiempos_ventanas,
    _alinear_spa,
    _calcular_mascaras_comunes,
    _canal_alineado_uv,
    _filtrar_pasa_altos_causal,
    _welch_por_bloques,
    _convertir_potencia_a_db,
    _ajustar_reconstruida_a_timeline,
    _suavizar_y_desplazar,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)


## 1. Registros de prueba

In [2]:

# Registros BIS que se van a comparar (unilaterales y bilaterales)
REGISTROS = [
    {
        "id": "L03041035",
        "modo": "unilateral",
        "base": Path(r"C:\Users\usuario\Downloads\data\data\data_bis_advanced\M-TA6m-03041035\DH03041035"),
    },
    {
        "id": "L06211051",
        "modo": "unilateral",
        "base": Path(r"C:\Users\usuario\OneDrive - Universidad de Burgos\Documentos\tfg\datos\pacientes\PACIENTE_002\SESIONES\SESION_002_L06211051\BIS\DH06211051"),
    },
    {
        "id": "L05061009",
        "modo": "bilateral",
        "base": Path(r"C:\Users\usuario\OneDrive - Universidad de Burgos\Documentos\tfg\datos\pacientes\PACIENTE_001\SESIONES\SESION_009_L05061009\BIS\DH05061009"),
    },
    {
        "id": "L04301310",
        "modo": "bilateral",
        "base": Path(r"C:\Users\usuario\OneDrive - Universidad de Burgos\Documentos\tfg\datos\pacientes\PACIENTE_001\SESIONES\SESION_001_L04301310\BIS\DH04301310"),
    },
]

# Nº máximo de segundos evaluados por registro, para acotar el coste computacional
MAX_SECONDS = 1800

def rutas_registro(reg):
    """
    Construye las rutas esperadas de los archivos BIS de un registro.

    Parámetros
    ----------
    reg : dict
        Debe contener "id" (str), "modo" ("unilateral"/"bilateral") y "base" (Path).

    Devuelve
    --------
    dict[str, Path]
        Rutas de .spa, .f_a, .h_a, .t_a y .r2a/.r4a (según el modo).
        Solo construye las rutas; no comprueba que los archivos existan.
    """
    base = reg["base"]
    sufijo_raw = ".r4a" if reg["modo"] == "bilateral" else ".r2a"  # bilateral -> .r4a, unilateral -> .r2a
    return {
        "spa": base / f"{reg['id']}.spa",
        "fa": base / f"{reg['id']}.f_a",
        "ha": base / f"{reg['id']}.h_a",
        "ta": base / f"{reg['id']}.t_a",
        "raw": base / f"{reg['id']}{sufijo_raw}",
    }

# Comprobación rápida: ¿existen todos los archivos esperados de cada registro?
tabla_registros = []
for reg in REGISTROS:
    rutas = rutas_registro(reg)
    tabla_registros.append({
        "registro": reg["id"],
        "modo": reg["modo"],
        "base": str(reg["base"]),
        **{nombre: ruta.exists() for nombre, ruta in rutas.items()},
    })

display(pd.DataFrame(tabla_registros))


,registro,modo,base,spa,fa,ha,ta,raw
0,L03041035,unilateral,C:\Users\usuario\Downloads\data\data\data_bis_...,True,True,True,True,True
1,L06211051,unilateral,C:\Users\usuario\OneDrive - Universidad de Bur...,True,True,True,True,True
2,L05061009,bilateral,C:\Users\usuario\OneDrive - Universidad de Bur...,True,True,True,True,True
3,L04301310,bilateral,C:\Users\usuario\OneDrive - Universidad de Bur...,True,True,True,True,True


## 2. Funciones auxiliares

## Documentación técnica de funciones auxiliares

Las funciones de este cuaderno son auxiliares para la validación experimental. No sustituyen a las funciones principales de la aplicación, sino que reutilizan sus parámetros y parte de su flujo para comparar tres formas de reconstruir la DSA: Welch, `spectrogram` y wavelet.

---

### `rutas_registro(reg)`

```python
rutas_registro(reg)
```

Construye las rutas esperadas de los archivos BIS de un registro.

**Parámetros**

| Parámetro | Tipo | Descripción |
|---|---|---|
| `reg` | `dict` | Diccionario con la información básica del registro. Debe contener `id`, `modo` y `base`. |
| `reg["id"]` | `str` | Identificador del registro BIS, por ejemplo `L03041035`. |
| `reg["modo"]` | `str` | Tipo de monitorización: `unilateral` o `bilateral`. |
| `reg["base"]` | `pathlib.Path` | Carpeta donde están los archivos exportados por el BIS. |

**Devuelve**

| Salida | Tipo | Descripción |
|---|---|---|
| `rutas` | `dict[str, Path]` | Diccionario con las rutas de `.spa`, `.f_a`, `.h_a`, `.t_a` y `.r2a`/`.r4a`. |

**Notas**

- Si el registro es unilateral, espera un archivo `.r2a`.
- Si el registro es bilateral, espera un archivo `.r4a`.
- La función solo construye rutas; no comprueba por sí misma que los archivos existan.

---

### `alinear_dsa_exportada(tiempo, dsa, timeline)`

```python
alinear_dsa_exportada(tiempo, dsa, timeline)
```

Alinea la matriz espectral original del archivo `.f_a` con la timeline común usada en la comparación.

**Parámetros**

| Parámetro | Tipo | Descripción |
|---|---|---|
| `tiempo` | `pd.Series` o array-like | Instantes temporales del `.f_a`. |
| `dsa` | `pd.DataFrame` | Matriz DSA exportada por el monitor. Las columnas corresponden a frecuencias. |
| `timeline` | `pd.Series`, `pd.DatetimeIndex` o array-like | Timeline común del registro, con resolución de un segundo. |

**Devuelve**

| Salida | Tipo | Descripción |
|---|---|---|
| `dsa_alineada` | `pd.DataFrame` | Matriz DSA reindexada sobre `timeline`, con una fila por segundo. |

**Comportamiento**

- Redondea los tiempos al segundo (`floor("s")`).
- Si hay segundos duplicados en el `.f_a`, conserva la última aparición.
- Los segundos presentes en `timeline` pero ausentes en `.f_a` quedan como `NaN`.

---

### `matriz_welch_db(senal_uv, header, parametros)`

```python
matriz_welch_db(senal_uv, header, parametros)
```

Reconstruye la DSA mediante Welch usando el flujo actual de la aplicación.

**Parámetros**

| Parámetro | Tipo | Descripción |
|---|---|---|
| `senal_uv` | `np.ndarray` | Señal EEG de un canal, alineada, filtrada y expresada en microvoltios. |
| `header` | `dict` | Parámetros extraídos de `.h_a`. Debe incluir `fs`. |
| `parametros` | `dict` | Parámetros de reconstrucción. Incluye ventana, avance, rango de frecuencias, paso frecuencial, modo Welch y referencia en dB. |

**Devuelve**

| Salida | Tipo | Descripción |
|---|---|---|
| `matriz_db` | `np.ndarray` | Matriz tiempo-frecuencia reconstruida en dB. Filas = ventanas; columnas = frecuencias. |
| `frecuencias` | `np.ndarray` | Frecuencias seleccionadas entre 0.5 y 30 Hz. |
| `tiempos_s` | `np.ndarray` | Tiempo, en segundos, asociado a cada ventana. En este flujo se usa la referencia temporal central. |

**Parámetros metodológicos usados**

- Ventana Hann.
- Ventana temporal de 2 s.
- Avance de 1 s.
- `detrend="constant"`.
- PSD en `uV²/Hz`.
- Rango de 0.5 a 30 Hz.
- Bins de 0.5 Hz.
- Conversión a dB mediante la función `_convertir_potencia_a_db`.

**Notas**

- Es el método de referencia de la aplicación.
- La PSD se integra por bin antes de la conversión a dB.

---

### `matriz_spectrogram_db(senal_uv, header, parametros)`

```python
matriz_spectrogram_db(senal_uv, header, parametros)
```

Reconstruye la DSA usando `scipy.signal.spectrogram`, manteniendo los mismos parámetros que Welch.

**Parámetros**

| Parámetro | Tipo | Descripción |
|---|---|---|
| `senal_uv` | `np.ndarray` | Señal EEG de un canal, alineada, filtrada y expresada en microvoltios. |
| `header` | `dict` | Cabecera del registro. Debe incluir `fs`. |
| `parametros` | `dict` | Parámetros de reconstrucción compartidos con Welch. |

**Devuelve**

| Salida | Tipo | Descripción |
|---|---|---|
| `matriz_db` | `np.ndarray` | Matriz tiempo-frecuencia reconstruida en dB. |
| `frecuencias` | `np.ndarray` | Frecuencias seleccionadas entre 0.5 y 30 Hz. |
| `tiempos_s` | `np.ndarray` | Tiempo, en segundos, asociado a cada ventana. |

**Parámetros metodológicos usados**

- `window="hann"`.
- `nperseg = 2 * fs`.
- `noverlap = nperseg - fs`, para conseguir avance de 1 s.
- `detrend="constant"`.
- `scaling="density"`.
- `mode="psd"`.
- `nfft = fs / paso_frecuencia`.

**Notas**

- Con estos parámetros, `spectrogram` y Welch pueden dar resultados idénticos o prácticamente idénticos.
- Se mantiene en el cuaderno para demostrar que no aporta una mejora real frente a Welch cuando se configura de forma equivalente.

---

### `matriz_wavelet_db(senal_uv, header, parametros)`

```python
matriz_wavelet_db(senal_uv, header, parametros)
```

Genera una representación tiempo-frecuencia mediante wavelet continua y la adapta a una matriz comparable con la DSA.

**Parámetros**

| Parámetro | Tipo | Descripción |
|---|---|---|
| `senal_uv` | `np.ndarray` | Señal EEG de un canal, alineada, filtrada y expresada en microvoltios. |
| `header` | `dict` | Cabecera del registro. Debe incluir `fs`. |
| `parametros` | `dict` | Parámetros generales del flujo: rango de frecuencias, paso frecuencial, ventana temporal y referencia en dB. |

**Devuelve**

| Salida | Tipo | Descripción |
|---|---|---|
| `matriz_db` | `np.ndarray` | Matriz tiempo-frecuencia expresada en dB tras resumir la energía wavelet por ventanas. |
| `frecuencias` | `np.ndarray` | Frecuencias evaluadas entre 0.5 y 30 Hz. |
| `tiempos_s` | `np.ndarray` | Tiempo central de cada ventana de 2 s. |

**Parámetros metodológicos usados**

- Wavelet compleja Morlet: `cmor1.5-1.0`.
- Frecuencias de 0.5 a 30 Hz con paso de 0.5 Hz.
- Potencia instantánea calculada como `abs(coeficientes) ** 2`.
- Promedio de potencia en ventanas de 2 s con avance de 1 s.

**Notas**

- Es un método exploratorio dentro del cuaderno.
- La energía wavelet no está calibrada exactamente igual que la PSD de Welch.
- Por eso puede mostrar correlaciones altas pero errores absolutos y sesgos elevados en dB.
- No se usa como método final de reconstrucción de la aplicación.

---

### `calcular_metricas_db(reconstruida, fa)`

```python
calcular_metricas_db(reconstruida, fa)
```

Calcula las métricas de comparación entre una DSA reconstruida y la DSA original del `.f_a`.

**Parámetros**

| Parámetro | Tipo | Descripción |
|---|---|---|
| `reconstruida` | `pd.DataFrame` | DSA reconstruida y ya alineada, suavizada, desplazada y enmascarada. |
| `fa` | `pd.DataFrame` | DSA original del `.f_a`, alineada y enmascarada con la misma máscara final. |

**Devuelve**

| Clave | Tipo | Descripción |
|---|---|---|
| `celdas_validas` | `int` | Número de celdas comunes válidas usadas para calcular las métricas. |
| `Pearson` | `float` | Correlación lineal entre ambas matrices. |
| `Spearman` | `float` | Correlación por rangos. |
| `MAE_dB` | `float` | Error absoluto medio en dB. |
| `RMSE_dB` | `float` | Raíz del error cuadrático medio en dB. |
| `bias_rec_menos_fa_dB` | `float` | Sesgo medio: reconstrucción menos `.f_a`, en dB. |

**Notas**

- No aplica z-score.
- Solo compara celdas válidas comunes.
- Si hay menos de tres celdas válidas, devuelve `NaN` en las métricas de correlación y error.
- Para decidir el método final se priorizan `MAE_dB`, `RMSE_dB` y sesgo, porque Pearson no detecta bien desplazamientos constantes de escala.

---


In [3]:

def alinear_dsa_exportada(tiempo, dsa, timeline):
    """
    Alinea la DSA original del .f_a con la timeline común (resolución de 1 s).

    Parámetros
    ----------
    tiempo : pd.Series o array-like
        Instantes temporales del .f_a.
    dsa : pd.DataFrame
        Matriz DSA exportada por el monitor (columnas = frecuencias).
    timeline : pd.Series, pd.DatetimeIndex o array-like
        Timeline común del registro, con resolución de 1 s.

    Devuelve
    --------
    pd.DataFrame
        DSA reindexada sobre timeline (una fila por segundo). Los segundos
        de timeline sin datos en .f_a quedan en NaN; si hay segundos
        duplicados en .f_a, se conserva la última aparición.
    """
    columnas = [float(c) for c in dsa.columns]
    salida = dsa.copy()
    salida.columns = columnas
    indice = pd.to_datetime(tiempo).dt.floor("s")  # redondeo al segundo
    salida.index = indice
    if salida.index.duplicated().any():
        salida = salida[~salida.index.duplicated(keep="last")]  # nos quedamos con la última muestra de cada segundo
    return salida.reindex(pd.DatetimeIndex(timeline)).reset_index(drop=True)


def matriz_welch_db(senal_uv, header, parametros):
    """
    Reconstruye la DSA mediante Welch (método de referencia de la aplicación).

    Parámetros
    ----------
    senal_uv : np.ndarray
        Señal EEG de un canal, alineada, filtrada y en microvoltios.
    header : dict
        Cabecera del registro; debe incluir "fs".
    parametros : dict
        Parámetros de reconstrucción (ventana, avance, rango de frecuencias,
        paso frecuencial, modo Welch, referencia dB, etc.).

    Devuelve
    --------
    matriz_db : np.ndarray
        Matriz tiempo-frecuencia en dB (filas = ventanas, columnas = frecuencias).
    frecuencias : np.ndarray
        Frecuencias entre 0.5 y 30 Hz.
    tiempos_s : np.ndarray
        Tiempo (s) de cada ventana (referencia temporal central).
    """
    potencia, frecuencias, tiempos_s = _welch_por_bloques(
        senal_uv,
        fs=header["fs"],
        ventana_seg=parametros["ventana_welch_s"],
        paso_seg=parametros["paso_welch_s"],
        fmin=parametros["fmin"],
        fmax=parametros["fmax"],
        paso_frecuencia=parametros["paso_frecuencia"],
        modo=parametros["modo_welch"],
        tiempo_referencia=parametros["tiempo_referencia"],
    )
    return _convertir_potencia_a_db(potencia, parametros), frecuencias, tiempos_s


def matriz_spectrogram_db(senal_uv, header, parametros):
    """
    Reconstruye la DSA con scipy.signal.spectrogram usando los mismos
    parámetros que Welch (ventana Hann, avance de 1 s, mismo rango y paso
    de frecuencias). Con esta configuración, spectrogram y Welch deberían
    dar resultados prácticamente idénticos.

    Parámetros
    ----------
    senal_uv : np.ndarray
        Señal EEG de un canal, alineada, filtrada y en microvoltios.
    header : dict
        Cabecera del registro; debe incluir "fs".
    parametros : dict
        Parámetros de reconstrucción compartidos con Welch.

    Devuelve
    --------
    matriz_db : np.ndarray
        Matriz tiempo-frecuencia en dB.
    frecuencias : np.ndarray
        Frecuencias entre 0.5 y 30 Hz.
    tiempos_s : np.ndarray
        Tiempo (s) de cada ventana.
    """
    fs = int(header["fs"])
    nperseg = int(parametros["ventana_welch_s"] * fs)
    paso = int(parametros["paso_welch_s"] * fs)
    noverlap = nperseg - paso  # para lograr un avance de 1 s entre ventanas
    nfft = int(fs / parametros["paso_frecuencia"])

    frecuencias, tiempos_s, psd = spectrogram(
        senal_uv,
        fs=fs,
        window="hann",
        nperseg=nperseg,
        noverlap=noverlap,
        nfft=nfft,
        detrend="constant",
        scaling="density",
        mode="psd",
    )
    mascara = (frecuencias >= parametros["fmin"]) & (frecuencias <= parametros["fmax"])  # nos quedamos solo con 0.5-30 Hz
    potencia = psd[mascara, :].T
    frecuencias = frecuencias[mascara]
    return _convertir_potencia_a_db(potencia, parametros), frecuencias, tiempos_s


def matriz_wavelet_db(senal_uv, header, parametros):
    """
    Genera una representación tiempo-frecuencia mediante wavelet continua
    (Morlet compleja) y la resume en ventanas de 2 s / avance 1 s para
    hacerla comparable con la DSA. Método exploratorio: la energía wavelet
    no está calibrada igual que la PSD de Welch, por lo que puede mostrar
    correlaciones altas pero sesgos y errores en dB elevados.

    Parámetros
    ----------
    senal_uv : np.ndarray
        Señal EEG de un canal, alineada, filtrada y en microvoltios.
    header : dict
        Cabecera del registro; debe incluir "fs".
    parametros : dict
        Parámetros generales del flujo (rango de frecuencias, paso
        frecuencial, ventana temporal, referencia en dB).

    Devuelve
    --------
    matriz_db : np.ndarray
        Matriz tiempo-frecuencia en dB (energía wavelet promediada por ventana).
    frecuencias : np.ndarray
        Frecuencias evaluadas entre 0.5 y 30 Hz.
    tiempos_s : np.ndarray
        Tiempo central (s) de cada ventana de 2 s.
    """
    fs = int(header["fs"])
    ventana = int(parametros["ventana_welch_s"] * fs)
    paso = int(parametros["paso_welch_s"] * fs)
    frecuencias = np.arange(
        parametros["fmin"],
        parametros["fmax"] + parametros["paso_frecuencia"] / 2,
        parametros["paso_frecuencia"],
    )

    wavelet = "cmor1.5-1.0"  # wavelet compleja Morlet
    central = pywt.central_frequency(wavelet)
    scales = central * fs / frecuencias  # escala equivalente a cada frecuencia deseada

    coef, _ = pywt.cwt(
        senal_uv,
        scales,
        wavelet,
        sampling_period=1 / fs,
        method="fft",
    )
    potencia_inst = np.abs(coef) ** 2  # potencia instantánea por muestra y frecuencia

    inicios = np.arange(0, len(senal_uv) - ventana + 1, paso, dtype=np.int64)
    potencia = np.empty((len(inicios), len(frecuencias)), dtype=float)
    for i, inicio in enumerate(inicios):
        potencia[i, :] = np.nanmean(potencia_inst[:, inicio:inicio + ventana], axis=1)  # potencia media dentro de cada ventana

    tiempos_s = (inicios + ventana / 2) / fs  # tiempo central de cada ventana
    return _convertir_potencia_a_db(potencia, parametros), frecuencias, tiempos_s


# Diccionario que asocia cada método con su función de reconstrucción,
# para poder recorrerlos todos igual en evaluar_registro()
METODOS = {
    "Welch": matriz_welch_db,
    "Spectrogram": matriz_spectrogram_db,
    "Wavelet": matriz_wavelet_db,
}


def calcular_metricas_db(reconstruida, fa):
    """
    Calcula las métricas de comparación entre una DSA reconstruida y la
    DSA original del .f_a (en dB, sin z-score).

    Parámetros
    ----------
    reconstruida : pd.DataFrame
        DSA reconstruida, ya alineada, suavizada, desplazada y enmascarada.
    fa : pd.DataFrame
        DSA original del .f_a, alineada y con la misma máscara final.

    Devuelve
    --------
    dict
        celdas_validas, Pearson, Spearman, MAE_dB, RMSE_dB y
        bias_rec_menos_fa_dB (reconstrucción - .f_a). Si hay menos de
        3 celdas válidas comunes, las métricas de correlación/error son NaN.
    """
    a = reconstruida.to_numpy(dtype=float).ravel()
    b = fa.to_numpy(dtype=float).ravel()
    mask = np.isfinite(a) & np.isfinite(b)  # solo celdas válidas en ambas matrices
    n = int(mask.sum())
    if n < 3:
        return {
            "celdas_validas": n,
            "Pearson": np.nan,
            "Spearman": np.nan,
            "MAE_dB": np.nan,
            "RMSE_dB": np.nan,
            "bias_rec_menos_fa_dB": np.nan,
        }
    diff = a[mask] - b[mask]
    return {
        "celdas_validas": n,
        "Pearson": pearsonr(a[mask], b[mask]).statistic,
        "Spearman": spearmanr(a[mask], b[mask]).correlation,
        "MAE_dB": float(np.mean(np.abs(diff))),
        "RMSE_dB": float(np.sqrt(np.mean(diff ** 2))),
        "bias_rec_menos_fa_dB": float(np.mean(diff)),  # positivo = reconstrucción por encima del .f_a
    }


## 3. Preparación común y evaluación

### Documentación técnica: preparación y evaluación de registros

### `preparar_registro(reg)`

```python
preparar_registro(reg)
```

Lee, alinea y prepara un registro BIS para poder comparar métodos espectrales.

**Parámetros**

| Parámetro | Tipo | Descripción |
|---|---|---|
| `reg` | `dict` | Diccionario con `id`, `modo` y `base`. |

**Devuelve**

| Clave | Tipo | Descripción |
|---|---|---|
| `reg` | `dict` | Información original del registro. |
| `rutas` | `dict[str, Path]` | Rutas de los archivos usados. |
| `df_spa` | `pd.DataFrame` | Variables procesadas leídas desde `.spa`. |
| `header` | `dict` | Parámetros de adquisición de `.h_a`. |
| `raw` | `np.ndarray` | Onda cruda leída desde `.r2a` o `.r4a`. |
| `timeline` | `pd.Series` | Timeline común entre raw, `.spa` y `.f_a`. |
| `info` | `dict` | Información de alineación entre raw y timeline común. |
| `parametros` | `dict` | Parámetros de reconstrucción actualizados con `LoFilter`. |
| `suavizado_s` | `int` | Ventana de suavizado extraída de `SpSmooth`. |
| `codigo_spsmooth` | `int` o similar | Código original de `SpSmooth` en el `.spa`. |
| `codigo_lofilter` | `int` o `None` | Código de `LoFilter` si existe. |
| `filtro_hz` | `float` | Frecuencia de corte del filtro pasa-altos. |
| `mascaras` | `dict` | Máscara previa al suavizado y máscara final por lado. |
| `fa_por_lado` | `dict[str, pd.DataFrame]` | DSA original alineada por lado. |
| `canales` | `dict[str, int]` | Canal raw utilizado por cada lado. |

**Proceso interno**

1. Carga `.spa`.
2. Lee `.h_a`, `.t_a` y la onda cruda.
3. Lee los tiempos del `.f_a`.
4. Calcula la timeline común.
5. Recorta a `MAX_SECONDS` para limitar el coste computacional.
6. Extrae `LoFilter` y `SpSmooth`.
7. Calcula máscaras comunes de calidad.
8. Carga la DSA original del `.f_a` y la alinea a la timeline.
9. Define los canales:
   - unilateral: canal 1;
   - bilateral: canal 1 para izquierda y canal 3 para derecha.

**Notas**

- Esta función concentra la parte común del flujo para que la comparación sea justa.
- Los tres métodos espectrales reciben exactamente la misma señal preparada.

---

### `evaluar_registro(reg)`

```python
evaluar_registro(reg)
```

Ejecuta la comparación completa de métodos para un registro BIS.

**Parámetros**

| Parámetro | Tipo | Descripción |
|---|---|---|
| `reg` | `dict` | Registro BIS que se quiere evaluar. |

**Devuelve**

| Salida | Tipo | Descripción |
|---|---|---|
| `df_resultado` | `pd.DataFrame` | Tabla con una fila por registro, lado y método espectral. |

**Columnas principales del `DataFrame`**

| Columna | Descripción |
|---|---|
| `registro` | Identificador del registro BIS. |
| `modo` | `unilateral` o `bilateral`. |
| `lado` | `unilateral`, `izquierda` o `derecha`. |
| `metodo` | `Welch`, `Spectrogram` o `Wavelet`. |
| `segundos_timeline` | Duración evaluada en segundos. |
| `SpSmooth_s` | Ventana de suavizado aplicada. |
| `shift_s` | Desplazamiento temporal aplicado. |
| `LoFilter_Hz` | Frecuencia de corte del filtro pasa-altos. |
| `mascara_final_s` | Número de segundos enmascarados. |
| `celdas_validas` | Celdas usadas en la comparación. |
| `Pearson` | Correlación de Pearson. |
| `Spearman` | Correlación de Spearman. |
| `MAE_dB` | Error absoluto medio. |
| `RMSE_dB` | Raíz del error cuadrático medio. |
| `bias_rec_menos_fa_dB` | Sesgo medio reconstrucción - `.f_a`. |

**Proceso interno**

1. Prepara el registro con `preparar_registro`.
2. Extrae el canal raw correspondiente.
3. Aplica el filtro pasa-altos causal.
4. Reconstruye la DSA con cada método.
5. Ajusta la reconstrucción a la timeline común.
6. Aplica suavizado con `SpSmooth`.
7. Aplica el shift temporal.
8. Aplica la máscara final.
9. Calcula métricas frente al `.f_a`.

**Notas**

- Esta es la función que produce las filas de resultados del cuaderno.
- Permite comparar métodos manteniendo constantes el resto de decisiones metodológicas.


In [4]:

def preparar_registro(reg):
    """
    Lee, alinea y prepara un registro BIS para poder comparar métodos
    espectrales. Concentra la parte común del flujo (lectura, timeline,
    filtro, suavizado, máscaras) para que los tres métodos reciban
    exactamente la misma señal preparada.

    Parámetros
    ----------
    reg : dict
        Registro con "id", "modo" y "base".

    Devuelve
    --------
    dict
        Contiene, entre otras claves: reg, rutas, df_spa, header, raw,
        timeline, info, parametros (con LoFilter aplicado), suavizado_s,
        codigo_spsmooth, codigo_lofilter, filtro_hz, mascaras,
        fa_por_lado (DSA original alineada por lado) y canales
        (canal raw usado por cada lado).
    """
    rutas = rutas_registro(reg)
    modo = reg["modo"]

    # 1. Variables procesadas (.spa)
    df_spa = (
        cargar_spa_bilateral_desde_ruta(rutas["spa"])
        if modo == "bilateral"
        else cargar_spa_unilateral_desde_ruta(rutas["spa"])
    )

    # 2. Cabecera de adquisición, inicio temporal y onda cruda
    header = extraer_parametros_eeg_desde_ruta(rutas["ha"])
    inicio_raw = leer_inicio_ta_desde_ruta(rutas["ta"])
    raw = _leer_raw_intercalado_desde_ruta(rutas["raw"], header["num_canales"])

    # 3. Tiempos del .f_a
    tiempos_fa = cargar_tiempos_fa_desde_ruta(rutas["fa"])

    # 4. Timeline común entre raw, .spa y .f_a
    cobertura = calcular_timeline_comun(
        inicio_raw,
        len(raw),
        header["fs"],
        df_spa["Time"],
        tiempos_fa,
    )
    timeline = cobertura["timeline"].reset_index(drop=True)
    if MAX_SECONDS:
        # 5. Recorte a MAX_SECONDS para limitar el coste computacional
        timeline = timeline.iloc[:MAX_SECONDS].reset_index(drop=True)

    info = _calcular_indices_alineacion(
        inicio_raw,
        timeline,
        header["fs"],
        len(raw),
    )

    # 6. Filtro pasa-altos (LoFilter) y suavizado (SpSmooth) declarados en el .spa
    parametros = dict(PARAMETROS_RECONSTRUCCION)
    filtro_predeterminado = parametros.get(
        "filtro_pasa_altos_hz",
        parametros.get("filtro_pasa_altos_predeterminado_hz", 0.25),
    )
    codigo_lofilter, filtro_hz, _ = _extraer_filtro_lofilter(
        df_spa,
        filtro_predeterminado,
    )
    parametros["filtro_pasa_altos_hz"] = filtro_hz

    suavizado_s, codigo_spsmooth = _extraer_suavizado_spsmooth(df_spa)
    df_merge = _alinear_spa(timeline, df_spa)

    tiempos_ventanas = _calcular_tiempos_ventanas(
        info["muestras_objetivo"],
        header["fs"],
        parametros["ventana_welch_s"],
        parametros["paso_welch_s"],
        parametros["tiempo_referencia"],
    )

    # 7. Máscaras comunes de calidad (antes de suavizar y máscara final)
    mascaras = _calcular_mascaras_comunes(
        modo,
        raw,
        header,
        info,
        timeline,
        df_merge,
        tiempos_ventanas,
        suavizado_s,
        parametros,
        cobertura["tiempos_spa"],
        cobertura["tiempos_fa"],
    )

    # 8. DSA original (.f_a) alineada a la timeline, por lado
    if modo == "bilateral":
        tiempo_fa, frecuencias_fa, dsa_fa_izq, dsa_fa_der = cargar_fa_bilateral_completo_desde_ruta(rutas["fa"])
        fa_por_lado = {
            "izquierda": alinear_dsa_exportada(tiempo_fa, dsa_fa_izq, timeline),
            "derecha": alinear_dsa_exportada(tiempo_fa, dsa_fa_der, timeline),
        }
        # 9. Canal raw por lado: 1 (índice 0) izquierda, 3 (índice 2) derecha
        canales = {"izquierda": 0, "derecha": 2}
    else:
        tiempo_fa, frecuencias_fa, dsa_fa = cargar_fa_unilateral_desde_ruta(rutas["fa"])
        fa_por_lado = {
            "unilateral": alinear_dsa_exportada(tiempo_fa, dsa_fa, timeline),
        }
        # 9. Canal raw único: canal 1 (índice 0)
        canales = {"unilateral": 0}

    return {
        "reg": reg,
        "rutas": rutas,
        "df_spa": df_spa,
        "header": header,
        "raw": raw,
        "timeline": timeline,
        "info": info,
        "parametros": parametros,
        "suavizado_s": int(suavizado_s),
        "codigo_spsmooth": codigo_spsmooth,
        "codigo_lofilter": codigo_lofilter,
        "filtro_hz": filtro_hz,
        "mascaras": mascaras,
        "fa_por_lado": fa_por_lado,
        "canales": canales,
    }


def evaluar_registro(reg):
    """
    Ejecuta la comparación completa (Welch, Spectrogram, Wavelet) para un
    registro BIS: prepara el registro, reconstruye la DSA con cada método,
    la ajusta / suaviza / desplaza / enmascara igual que el flujo real, y
    calcula las métricas frente al .f_a.

    Parámetros
    ----------
    reg : dict
        Registro BIS a evaluar ("id", "modo", "base").

    Devuelve
    --------
    pd.DataFrame
        Una fila por combinación de lado y método, con las columnas
        descritas en la documentación técnica (registro, modo, lado,
        metodo, métricas, etc.).
    """
    preparado = preparar_registro(reg)
    filas = []
    shift_s = preparado["mascaras"]["shift_s"]

    for lado, canal in preparado["canales"].items():
        # 2. Canal raw correspondiente, ya escalado a microvoltios
        senal = _canal_alineado_uv(
            preparado["raw"],
            canal,
            preparado["header"]["pendiente"],
            preparado["header"]["offset"],
            preparado["info"],
        )
        # 3. Filtro pasa-altos causal (LoFilter)
        senal = _filtrar_pasa_altos_causal(
            senal,
            preparado["header"]["fs"],
            preparado["parametros"]["filtro_pasa_altos_hz"],
            preparado["parametros"]["orden_filtro_pasa_altos"],
        )

        mask_entrada = preparado["mascaras"]["entrada_suavizado"][lado].reset_index(drop=True)
        mask_final = preparado["mascaras"]["final"][lado].reset_index(drop=True)

        fa = preparado["fa_por_lado"][lado].copy()
        fa.loc[mask_final.to_numpy(), :] = np.nan  # aplicamos la misma máscara final al .f_a

        for metodo, funcion in METODOS.items():
            # 4. Reconstrucción de la DSA con el método actual
            matriz_db, frecuencias, tiempos_s = funcion(
                senal,
                preparado["header"],
                preparado["parametros"],
            )
            # 5. Ajuste de la reconstrucción a la timeline común
            reconstruida = _ajustar_reconstruida_a_timeline(
                matriz_db,
                frecuencias,
                tiempos_s,
                preparado["timeline"],
            )
            # 6-7. Suavizado (SpSmooth) y shift temporal del flujo actual
            reconstruida = _suavizar_y_desplazar(
                reconstruida,
                preparado["suavizado_s"],
                shift_s,
                mascara_inicial=mask_entrada,
            )
            # 8. Máscara final de calidad
            reconstruida.loc[mask_final.to_numpy(), :] = np.nan

            # 9. Métricas frente al .f_a
            metricas = calcular_metricas_db(reconstruida, fa)
            filas.append({
                "registro": reg["id"],
                "modo": reg["modo"],
                "lado": lado,
                "metodo": metodo,
                "segundos_timeline": len(preparado["timeline"]),
                "SpSmooth_s": preparado["suavizado_s"],
                "shift_s": shift_s,
                "LoFilter_Hz": preparado["filtro_hz"],
                "mascara_final_s": int(mask_final.sum()),
                **metricas,
            })

    return pd.DataFrame(filas)


## 4. Ejecutar comparación

In [5]:

# Evaluamos cada registro (los 3 métodos y todos sus lados) y concatenamos
# los resultados en un único DataFrame con una fila por registro/lado/método
resultados = []
for reg in REGISTROS:
    print(f"Evaluando {reg['id']} ({reg['modo']})...")
    resultados.append(evaluar_registro(reg))

df_resultados = pd.concat(resultados, ignore_index=True)

# Guardamos los resultados en CSV para poder revisarlos fuera del notebook
OUT_CSV = Path(r"C:\Users\usuario\Downloads\validacion_metodos_dsa_multiregistro_resultados.csv")
df_resultados.to_csv(OUT_CSV, index=False)

display(df_resultados.sort_values(["registro", "lado", "RMSE_dB"]))
print("CSV guardado en:", OUT_CSV)


Evaluando L03041035 (unilateral)...


Evaluando L06211051 (unilateral)...


Evaluando L05061009 (bilateral)...


Evaluando L04301310 (bilateral)...


,registro,modo,lado,metodo,segundos_timeline,SpSmooth_s,shift_s,LoFilter_Hz,mascara_final_s,celdas_validas,Pearson,Spearman,MAE_dB,RMSE_dB,bias_rec_menos_fa_dB
0,L03041035,unilateral,unilateral,Welch,1800,3,10,2.5,177,97380,0.492407,0.435153,5.256057,7.034714,2.888228
1,L03041035,unilateral,unilateral,Spectrogram,1800,3,10,2.5,177,97380,0.492407,0.435153,5.256057,7.034714,2.888228
2,L03041035,unilateral,unilateral,Wavelet,1800,3,10,2.5,177,97380,0.533335,0.469739,18.672028,19.520246,18.641783
15,L04301310,bilateral,derecha,Welch,1800,3,6,2.5,16,107040,0.889378,0.887830,3.014809,4.053186,0.045137
16,L04301310,bilateral,derecha,Spectrogram,1800,3,6,2.5,16,107040,0.889378,0.887830,3.014809,4.053186,0.045137
17,L04301310,bilateral,derecha,Wavelet,1800,3,6,2.5,16,107040,0.939717,0.951730,15.929842,16.169097,15.908830
12,L04301310,bilateral,izquierda,Welch,1800,3,6,2.5,11,107340,0.854341,0.876791,3.139515,4.690452,-0.097277
13,L04301310,bilateral,izquierda,Spectrogram,1800,3,6,2.5,11,107340,0.854341,0.876791,3.139515,4.690452,-0.097277
14,L04301310,bilateral,izquierda,Wavelet,1800,3,6,2.5,11,107340,0.897110,0.944267,15.891958,16.175250,15.732817
9,L05061009,bilateral,derecha,Welch,946,3,6,2.5,7,56340,0.872174,0.859571,2.834052,3.802971,-0.073122


CSV guardado en: C:\Users\usuario\Downloads\validacion_metodos_dsa_multiregistro_resultados.csv


## 5. Resumen

In [6]:

# Promedio de las métricas por método, a través de todos los registros y lados
resumen_metodo = (
    df_resultados
    .groupby("metodo", as_index=False)
    .agg(
        n_comparaciones=("registro", "count"),
        Pearson_medio=("Pearson", "mean"),
        Spearman_medio=("Spearman", "mean"),
        MAE_medio_dB=("MAE_dB", "mean"),
        RMSE_medio_dB=("RMSE_dB", "mean"),
        bias_medio_dB=("bias_rec_menos_fa_dB", "mean"),
    )
    .sort_values("RMSE_medio_dB")  # RMSE es el criterio principal para decidir el método final
)

display(Markdown("### Promedio global por método"))
display(resumen_metodo)

# Para cada registro y lado, nos quedamos con el método que obtiene menor RMSE
mejor_por_lado = (
    df_resultados
    .sort_values(["registro", "lado", "RMSE_dB"])
    .groupby(["registro", "lado"], as_index=False)
    .first()
)

display(Markdown("### Mejor método por registro y lado"))
display(mejor_por_lado[[
    "registro", "modo", "lado", "metodo", "Pearson", "MAE_dB", "RMSE_dB", "bias_rec_menos_fa_dB"
]])


### Promedio global por método

,metodo,n_comparaciones,Pearson_medio,Spearman_medio,MAE_medio_dB,RMSE_medio_dB,bias_medio_dB
0,Spectrogram,6,0.805434,0.797250,3.454749,4.836533,0.611765
2,Welch,6,0.805434,0.797250,3.454749,4.836533,0.611765
1,Wavelet,6,0.851939,0.860746,16.431503,16.781361,16.337548


### Mejor método por registro y lado

,registro,modo,lado,metodo,Pearson,MAE_dB,RMSE_dB,bias_rec_menos_fa_dB
0,L03041035,unilateral,unilateral,Welch,0.492407,5.256057,7.034714,2.888228
1,L04301310,bilateral,derecha,Welch,0.889378,3.014809,4.053186,0.045137
2,L04301310,bilateral,izquierda,Welch,0.854341,3.139515,4.690452,-0.097277
3,L05061009,bilateral,derecha,Welch,0.872174,2.834052,3.802971,-0.073122
4,L05061009,bilateral,izquierda,Welch,0.863195,2.935059,4.387799,-0.178087
5,L06211051,unilateral,unilateral,Welch,0.861111,3.549002,5.050074,1.085712


## Resumen

Welch y `spectrogram` pueden salir muy parecidos porque, con los mismos parámetros, ambos construyen un mapa tiempo-frecuencia basado en ventanas Hann, PSD y solapamiento temporal.

La wavelet se incluye como alternativa exploratoria. Su energía no queda calibrada exactamente igual que la PSD en `uV²/Hz` usada por Welch/spectrogram y por eso no debe interpretarse solo por Pearson: la escala absoluta en dB, el sesgo, el MAE y el RMSE son los criterios importantes.